# Cross-Country Comparison: Solar Data Analysis

## Overview
This notebook compares solar farm data across Benin, Sierra Leone, and Togo to identify relative solar potential and key differences.

## Objectives
1. Load cleaned datasets from all three countries
2. Compare key metrics (GHI, DNI, DHI) using boxplots
3. Create summary statistics table
4. Perform statistical testing (ANOVA/Kruskal-Wallis)
5. Identify key observations and insights
6. Create visual summaries and rankings


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, kruskal
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


## 1. Data Loading


In [ ]:
# Load cleaned datasets
benin_df = pd.read_csv('../data/benin_clean.csv')
sierra_leone_df = pd.read_csv('../data/sierra_leone_clean.csv')
togo_df = pd.read_csv('../data/togo_clean.csv')

# Add country column for identification
benin_df['Country'] = 'Benin'
sierra_leone_df['Country'] = 'Sierra Leone'
togo_df['Country'] = 'Togo'

# Convert Timestamp to datetime
benin_df['Timestamp'] = pd.to_datetime(benin_df['Timestamp'])
sierra_leone_df['Timestamp'] = pd.to_datetime(sierra_leone_df['Timestamp'])
togo_df['Timestamp'] = pd.to_datetime(togo_df['Timestamp'])

# Combine all datasets
all_countries_df = pd.concat([benin_df, sierra_leone_df, togo_df], ignore_index=True)

print("Data loaded successfully!")
print(f"Benin shape: {benin_df.shape}")
print(f"Sierra Leone shape: {sierra_leone_df.shape}")
print(f"Togo shape: {togo_df.shape}")
print(f"Combined shape: {all_countries_df.shape}")


## 2. Metric Comparison (Boxplots)


In [ ]:
# Boxplots for GHI, DNI, and DHI
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# GHI comparison
sns.boxplot(data=all_countries_df, x='Country', y='GHI', ax=axes[0])
axes[0].set_title('GHI Comparison Across Countries')
axes[0].set_ylabel('GHI (W/m²)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# DNI comparison
sns.boxplot(data=all_countries_df, x='Country', y='DNI', ax=axes[1])
axes[1].set_title('DNI Comparison Across Countries')
axes[1].set_ylabel('DNI (W/m²)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

# DHI comparison
sns.boxplot(data=all_countries_df, x='Country', y='DHI', ax=axes[2])
axes[2].set_title('DHI Comparison Across Countries')
axes[2].set_ylabel('DHI (W/m²)')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Summary Statistics Table


In [ ]:
# Create summary statistics table
metrics = ['GHI', 'DNI', 'DHI']
countries = ['Benin', 'Sierra Leone', 'Togo']
country_dfs = {'Benin': benin_df, 'Sierra Leone': sierra_leone_df, 'Togo': togo_df}

summary_stats = []

for metric in metrics:
    for country in countries:
        df = country_dfs[country]
        if metric in df.columns:
            summary_stats.append({
                'Metric': metric,
                'Country': country,
                'Mean': df[metric].mean(),
                'Median': df[metric].median(),
                'Std Dev': df[metric].std(),
                'Min': df[metric].min(),
                'Max': df[metric].max()
            })

summary_df = pd.DataFrame(summary_stats)
print("Summary Statistics Table:")
print(summary_df.to_string(index=False))

# Pivot table for easier comparison
pivot_mean = summary_df.pivot(index='Country', columns='Metric', values='Mean')
pivot_median = summary_df.pivot(index='Country', columns='Metric', values='Median')
pivot_std = summary_df.pivot(index='Country', columns='Metric', values='Std Dev')

print("\n\nMean Values:")
print(pivot_mean.round(2))
print("\n\nMedian Values:")
print(pivot_median.round(2))
print("\n\nStandard Deviation:")
print(pivot_std.round(2))


## 4. Statistical Testing (ANOVA/Kruskal-Wallis)


In [ ]:
# Prepare data for statistical testing
ghi_benin = benin_df['GHI'].dropna()
ghi_sierra_leone = sierra_leone_df['GHI'].dropna()
ghi_togo = togo_df['GHI'].dropna()

dni_benin = benin_df['DNI'].dropna()
dni_sierra_leone = sierra_leone_df['DNI'].dropna()
dni_togo = togo_df['DNI'].dropna()

dhi_benin = benin_df['DHI'].dropna()
dhi_sierra_leone = sierra_leone_df['DHI'].dropna()
dhi_togo = togo_df['DHI'].dropna()

# Test normality (Shapiro-Wilk test on samples)
from scipy.stats import shapiro

def test_normality(data, name):
    """Test normality on a sample of data"""
    sample = data.sample(min(5000, len(data)), random_state=42) if len(data) > 5000 else data
    stat, p_value = shapiro(sample)
    return p_value > 0.05  # Return True if normally distributed

# Check normality for each metric
ghi_normal = all([test_normality(ghi_benin, 'GHI Benin'),
                  test_normality(ghi_sierra_leone, 'GHI Sierra Leone'),
                  test_normality(ghi_togo, 'GHI Togo')])

print("Normality Tests (Shapiro-Wilk on samples):")
print(f"GHI distributions appear normal: {ghi_normal}")

# Perform statistical tests
print("\n" + "="*50)
print("Statistical Tests for GHI:")
print("="*50)

# One-way ANOVA (if data is approximately normal)
if ghi_normal:
    f_stat, p_value_anova = f_oneway(ghi_benin, ghi_sierra_leone, ghi_togo)
    print(f"\nOne-way ANOVA:")
    print(f"F-statistic: {f_stat:.4f}")
    print(f"P-value: {p_value_anova:.6f}")
    if p_value_anova < 0.05:
        print("Result: Significant difference between countries (p < 0.05)")
    else:
        print("Result: No significant difference between countries (p >= 0.05)")

# Kruskal-Wallis test (non-parametric alternative)
h_stat_ghi, p_value_kw_ghi = kruskal(ghi_benin, ghi_sierra_leone, ghi_togo)
print(f"\nKruskal-Wallis Test (non-parametric):")
print(f"H-statistic: {h_stat_ghi:.4f}")
print(f"P-value: {p_value_kw_ghi:.6f}")
if p_value_kw_ghi < 0.05:
    print("Result: Significant difference between countries (p < 0.05)")
else:
    print("Result: No significant difference between countries (p >= 0.05)")

# Repeat for DNI and DHI
print("\n" + "="*50)
print("Statistical Tests for DNI:")
print("="*50)
h_stat_dni, p_value_kw_dni = kruskal(dni_benin, dni_sierra_leone, dni_togo)
print(f"Kruskal-Wallis H-statistic: {h_stat_dni:.4f}")
print(f"P-value: {p_value_kw_dni:.6f}")
if p_value_kw_dni < 0.05:
    print("Result: Significant difference between countries (p < 0.05)")
else:
    print("Result: No significant difference between countries (p >= 0.05)")

print("\n" + "="*50)
print("Statistical Tests for DHI:")
print("="*50)
h_stat_dhi, p_value_kw_dhi = kruskal(dhi_benin, dhi_sierra_leone, dhi_togo)
print(f"Kruskal-Wallis H-statistic: {h_stat_dhi:.4f}")
print(f"P-value: {p_value_kw_dhi:.6f}")
if p_value_kw_dhi < 0.05:
    print("Result: Significant difference between countries (p < 0.05)")
else:
    print("Result: No significant difference between countries (p >= 0.05)")


### Key Observations:

1. **GHI Comparison**: 
   - [Country X] shows the highest median GHI: [value] W/m²
   - [Country Y] shows the lowest median GHI: [value] W/m²
   - Variability: [Country Z] shows the greatest variability (std dev: [value])

2. **DNI Comparison**:
   - [Country X] has the highest direct normal irradiance
   - [Observations about DNI patterns]

3. **DHI Comparison**:
   - [Country X] has the highest diffuse horizontal irradiance
   - [Observations about DHI patterns]

4. **Statistical Significance**:
   - GHI: P-value = [value] - [Significant/Not significant]
   - DNI: P-value = [value] - [Significant/Not significant]
   - DHI: P-value = [value] - [Significant/Not significant]

5. **Solar Potential Ranking**:
   - Based on average GHI: 1) [Country], 2) [Country], 3) [Country]
   - Based on consistency (low std dev): 1) [Country], 2) [Country], 3) [Country]


## 6. Visual Summary - Country Rankings


In [ ]:
# Calculate average GHI by country
avg_ghi = all_countries_df.groupby('Country')['GHI'].mean().sort_values(ascending=False)

# Bar chart ranking countries by average GHI
plt.figure(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = plt.bar(avg_ghi.index, avg_ghi.values, color=colors, alpha=0.8, edgecolor='black')
plt.title('Average GHI by Country', fontsize=16, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Average GHI (W/m²)', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("Country Ranking by Average GHI:")
for idx, (country, value) in enumerate(avg_ghi.items(), 1):
    print(f"{idx}. {country}: {value:.2f} W/m²")


In [ ]:
# Additional visualizations: Violin plots for better distribution understanding
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# GHI violin plot
sns.violinplot(data=all_countries_df, x='Country', y='GHI', ax=axes[0])
axes[0].set_title('GHI Distribution Comparison')
axes[0].set_ylabel('GHI (W/m²)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# DNI violin plot
sns.violinplot(data=all_countries_df, x='Country', y='DNI', ax=axes[1])
axes[1].set_title('DNI Distribution Comparison')
axes[1].set_ylabel('DNI (W/m²)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

# DHI violin plot
sns.violinplot(data=all_countries_df, x='Country', y='DHI', ax=axes[2])
axes[2].set_title('DHI Distribution Comparison')
axes[2].set_ylabel('DHI (W/m²)')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Summary and Recommendations

### Key Findings:
1. **Highest Solar Potential**: [Country] shows the highest average GHI
2. **Most Consistent**: [Country] shows the most consistent solar irradiance
3. **Statistical Significance**: [Summary of statistical test results]

### Recommendations:
- **Primary Investment Target**: [Country] - [Reasoning]
- **Secondary Target**: [Country] - [Reasoning]
- **Risk Considerations**: [Any variability or consistency concerns]

### Next Steps:
- [Additional analyses recommended]
- [Data collection improvements]
- [Further investigation areas]
